# 07 - Consolidation and Export

## Role of This Notebook
This notebook builds the final artifact that goes to Grasshopper: a CSV unified by `ROW_ID`, with base variables, original spatial class, target cluster, predicted cluster, and top 5 species ranking.

## Closing Criterion
The final result must be legible, traceable, and useful outside Python. For that reason, the export preserves both the model prediction and the justifications by species.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#777777',
    'axes.grid': True,
    'grid.color': '#e6e6e6',
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})

ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = ROOT / 'data' / 'processed'
EXPORT_DIR = ROOT / 'exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

geometry = pd.read_csv(PROCESSED_DIR / 'geometry_labeled.csv')
predictions = pd.read_csv(PROCESSED_DIR / 'geometry_predictions.csv')
recommendations = pd.read_csv(PROCESSED_DIR / 'tile_plant_recommendations_intermediate.csv')

geometry.head()

## 1. Consolidation
The join is done by `ROW_ID` because it remains the unique key of the project. `TILE_ID` is kept as a useful spatial reference for Grasshopper, but the technical assembly rests on row identity.


In [ ]:
prediction_columns = [
    'ROW_ID', 'TILE_ID', 'spatial_label', 'plant_cluster_target', 'plant_cluster_target_label',
    'plant_cluster_pred', 'plant_cluster_pred_label', 'plant_cluster_pred_probability',
    'bridge_top1_probability', 'bridge_margin'
] + [col for col in predictions.columns if col.startswith('pred_cluster_') and col.endswith('_probability')]

final_df = geometry.merge(
    predictions[prediction_columns],
    on=['ROW_ID', 'TILE_ID', 'spatial_label'],
    how='left'
).merge(
    recommendations.drop(columns=['TILE_ID', 'spatial_label']),
    on='ROW_ID',
    how='left'
)

for col in ['plant_cluster_target_label', 'plant_cluster_pred_label']:
    x_col = f'{col}_x'
    y_col = f'{col}_y'
    if x_col in final_df.columns and y_col in final_df.columns:
        final_df[col] = final_df[x_col].fillna(final_df[y_col])
        final_df = final_df.drop(columns=[x_col, y_col])

final_df.head()

## 2. Final Export
The exported CSV is the project's main operational result. It preserves traceability from geometry to botanical recommendation, so it is useful both for academic inspection and for a possible later integration.


In [ ]:
export_path = EXPORT_DIR / 'tile_plant_recommendations.csv'
final_df.to_csv(export_path, index=False)
print('Saved export:', export_path)
print(f'Rows exported: {len(final_df):,}')
print(f'Columns exported: {len(final_df.columns)}')

## 3. Final Visual Summary
The visual closing section helps verify that the export preserves a reasonable distribution of clusters and that the new botanical layer does not completely lose its relationship with the original spatial class.


In [ ]:
pred_cluster_counts = final_df['plant_cluster_pred_label'].value_counts().sort_index()
spatial_counts = final_df['spatial_label'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(pred_cluster_counts.index, pred_cluster_counts.values, color='#4e79a7', edgecolor='#444444', linewidth=0.6)
axes[0].set_title('Predicted plant cluster distribution')
axes[0].set_xlabel('Predicted plant cluster')
axes[0].set_ylabel('Number of rows')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(spatial_counts.index, spatial_counts.values, color='#59a14f', edgecolor='#444444', linewidth=0.6)
axes[1].set_title('Original spatial label distribution')
axes[1].set_xlabel('Spatial label')
axes[1].set_ylabel('Number of rows')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

summary_df = pd.DataFrame({
    'metric': ['rows_exported', 'n_predicted_clusters', 'mean_pred_probability', 'mean_top1_score'],
    'value': [
        len(final_df),
        final_df['plant_cluster_pred_label'].nunique(),
        round(final_df['plant_cluster_pred_probability'].mean(), 4),
        round(final_df['top1_score'].mean(), 4)
    ]
})
display(summary_df)

### Conclusion of the Visual Closing Section
The final file consolidates `43100` rows and `55` columns, so it already works as a complete operational artifact. The final distribution keeps the dominance of `bridge_cluster_2` with `35018` rows, while `bridge_cluster_0` and `bridge_cluster_1` cover `6756` and `1326` rows respectively. In addition, the average predicted probability is very high (`0.9974`), and the average `top1_score` is `98.98`, suggesting that the pipeline not only classifies well, but also delivers recommendations with a sufficiently strong hierarchy for export and external use.


## 4. Preview of Critical Columns
This final preview helps verify that the export kept the most important information: the original spatial class, the plant cluster, the model prediction, and the recommended species with their reasons.


In [ ]:
preview_columns = [
    'TILE_ID', 'spatial_label', 'plant_cluster_target_label', 'plant_cluster_pred_label', 'plant_cluster_pred_probability',
    'top1_species', 'top1_score', 'top1_reason',
    'top2_species', 'top2_score', 'top2_reason'
]
display(final_df[preview_columns].head(10))